# meta-agent-gym: GRPO Training (Colab T4)

Train a policy to generate AGENT.md files using GRPO with Unsloth 4-bit LoRA.

**Steps:**
1. Clean environment + install deps (fails loudly if setup breaks)
2. Clone repo
3. Collect baselines (random + heuristic)
4. Run expert benchmark (upper bound)
5. GRPO training — dry-run then real
6. Evaluate trained model (LoRA adapter rollouts)
7. Generate monitoring plots
8. Generate comparison plots (baseline vs trained)
9. Package + download results

**Policy: fail loud, no mock fallbacks.** Any dependency error, missing file, or training failure stops the notebook so you can see and fix it — rather than generating plausible-looking placeholder output.

**Key install fix:** xformers is pinned to a prebuilt wheel version and installed with `--no-deps` to avoid the source build that was failing on Colab.

In [ ]:
#@title 1. Clean environment + Setup (run FIRST)
#@markdown Pins aligned to Unsloth 2026.4.8 requirements. Fails loudly if any import is broken.

import torch
import os
import sys

# Fail fast if no GPU
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. Requires a GPU runtime: "
        "Runtime -> Change runtime type -> T4 GPU (or better)."
    )
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Remove anything that could conflict
!pip uninstall -y trl unsloth unsloth_zoo pydantic xformers transformers datasets 2>/dev/null

# Pydantic (TRL <-> pydantic conflict pinned per commit 802e79e)
!pip install -q "pydantic>=2.0,<2.11"

# Core ML stack — versions aligned to Unsloth 2026.4.8 requirements
# unsloth requires: transformers !=5.0.0,<=5.5.0,>=4.51.3 and datasets !=4.0.*,<4.4.0,>=3.4.1
!pip install -q "transformers>=4.51.3,<5.0.0" "datasets>=3.4.1,<4.0.0" "accelerate>=0.24.0" "peft>=0.10.0" "bitsandbytes>=0.43.0"

# xformers: prebuilt wheel (avoids source build that was failing on Colab)
!pip install -q --no-deps "xformers==0.0.29.post3"

# TRL — Unsloth 2026.4.8 requires >=0.18.2,<=0.24.0 (excludes 0.19.0)
!pip install -q "trl>=0.18.2,!=0.19.0,<=0.24.0"

# Unsloth — --no-deps prevents it from dragging xformers-from-source
!pip install -q --no-deps "unsloth_zoo" "hf_transfer" "tyro"
!pip install -q --no-deps "unsloth"

# Remaining deps
!pip install -q "httpx>=0.24.0" "matplotlib>=3.7.0" "pandas>=1.5.0" "numpy>=1.24.0" "pyyaml>=6.0" "sentencepiece" "protobuf"

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# --- Unsloth wants to be imported BEFORE trl/transformers/peft for full patching ---
import unsloth  # noqa: F401 — order matters for the optimization patches
print(f"[OK] Unsloth {getattr(unsloth, '__version__', 'unknown')} imported first")

import pydantic
print(f"[OK] Pydantic {pydantic.__version__}")

from trl import GRPOConfig, GRPOTrainer  # noqa: F401
print("[OK] TRL GRPOTrainer imported")

import xformers
print(f"[OK] xformers {xformers.__version__}")

import transformers
print(f"[OK] transformers {transformers.__version__}")

import datasets
print(f"[OK] datasets {datasets.__version__}")

print("\n=== Setup complete — all dependencies verified ===")

In [ ]:
#@title 2. Clone repo + install openenv-core only

REPO_URL = 'https://github.com/Kaviyamurugadass/meta-agent-gym.git' #@param {type:'string'}

import os, sys

print(f'[...] Cloning {REPO_URL}')
!git clone {REPO_URL} /content/openenv-agent-gym

sys.path.insert(0, '/content/openenv-agent-gym')
os.chdir('/content/openenv-agent-gym')

# Install openenv-core explicitly (not the full project) so models.py gets the
# proper base classes. We SKIP `pip install -e .` because it pulls in project
# deps that conflict with cell 1's carefully pinned versions (transformers
# 5.0.0, datasets 4.0.0, trl 0.16.1 etc.). The code runs fine from sys.path.
!pip install -q "openenv-core==0.2.1" "fastapi" "uvicorn[standard]" "websockets"

print('[...] Verifying imports...')
from server.environment import Environment  # noqa: F401
print('[OK] Environment imported')

from training.rollout_collection import collect  # noqa: F401
print('[OK] Rollout collection imported')

# Sanity-check: does Observation have .done? (regression guard for earlier bug)
from models import Observation
assert hasattr(Observation(task_id='x', step=0, max_steps=1), 'done'), \
    "Observation.done missing â€” models.py fix not applied. Re-pull the repo."
print('[OK] Observation.done present')

print('\n=== Project loaded and ready ===')

In [ ]:
#@title 3. Collect baselines (random + heuristic)

import sys, os
sys.path.insert(0, '/content/openenv-agent-gym')
os.chdir('/content/openenv-agent-gym')

from training.rollout_collection import collect

print('Collecting random baseline (20 episodes)...')
random_ds = collect(episodes=20, policy='random', output_dir='data/baseline/random', seed=42)
print(f'Random: {random_ds.summary()}')

print('\nCollecting heuristic baseline (20 episodes)...')
heuristic_ds = collect(episodes=20, policy='heuristic', output_dir='data/baseline/heuristic', seed=42)
print(f'Heuristic: {heuristic_ds.summary()}')

In [ ]:
#@title 4. Run expert benchmark (upper bound)

import sys, os
sys.path.insert(0, '/content/openenv-agent-gym')
os.chdir('/content/openenv-agent-gym')

from training.benchmark import run_all

print('Running expert benchmark...')
results = run_all()
for r in results:
    print(f'{r.scenario_name:30s} reward={r.total_reward:7.3f} steps={r.steps_taken:3d} success={r.success}')

successes = [r for r in results if r.success]
if not successes:
    raise RuntimeError(
        "No expert scenarios succeeded â€” check scenarios/environment before training. "
        "Training with a broken environment wastes compute."
    )
expert_mean = sum(r.total_reward for r in successes) / len(successes)
print(f'\nExpert upper bound: mean_reward={expert_mean:.3f} ({len(successes)}/{len(results)} scenarios succeeded)')

In [ ]:
#@title Debug toggles (optional) — show model outputs in Cell 5
import os

# Print the first few raw model completions from reward_fn
os.environ["SHOW_COMPLETIONS"] = "1"
os.environ["SHOW_COMPLETIONS_LIMIT"] = "4"

# If you want training to crash on the first reward_fn error (to see traceback), uncomment:
# os.environ["REWARD_FN_FAIL_FAST"] = "1"

In [ ]:
#@title 5. GRPO Training (dry-run first, then real)

import sys, os, json
sys.path.insert(0, '/content/openenv-agent-gym')
os.chdir('/content/openenv-agent-gym')

MODEL_ID = 'Qwen/Qwen3-1.7B' #@param ['Qwen/Qwen3-1.7B', 'Qwen/Qwen2.5-0.5B', 'Qwen/Qwen2-0.5B']
NUM_EPOCHS = 2 #@param {type:'integer'}
NUM_GENERATIONS = 2 #@param {type:'integer'}
DATASET_EPISODES = 25 #@param {type:'integer'}

# Re-verify critical deps (cell 1 already checked, but this cell may be re-run standalone)
from trl import GRPOConfig, GRPOTrainer  # noqa: F401
import unsloth  # noqa: F401
from training.grpo_unsloth import main

base_argv = [
    'grpo_unsloth.py',
    '--model-id', MODEL_ID,
    '--num-epochs', str(NUM_EPOCHS),
    '--num-generations', str(NUM_GENERATIONS),
    '--dataset-episodes', str(DATASET_EPISODES),
    '--max-seq-length', '768',  # T4 memory constraint
    '--max-completion-length', '256',  # keep completions short so arrays don't get cut mid-JSON
    '--per-device-train-batch-size', '1',
    '--gradient-accumulation-steps', '4',
    '--learning-rate', '5e-6',
    '--output-dir', 'training/grpo-unsloth-output',
    '--seed', '42',
]

# Step A: Dry-run — validates config and imports without burning GPU time
print('=== Step A: Dry-run validation ===')
sys.argv = base_argv + ['--dry-run']
main()
print('[OK] Dry-run passed\n')

# Step B: Real training
print('=== Step B: Real GRPO training ===')
sys.argv = base_argv  # no --dry-run
main()
print('[OK] Training complete')

# Real-training sentinel — lets downstream cells and post-hoc checks distinguish
# this run from a mock/replay. If this file is missing, assume training did NOT run.
os.makedirs('training/grpo-unsloth-output', exist_ok=True)
summary_path = 'training/grpo-unsloth-output/training_summary.json'
with open(summary_path, 'w') as f:
    json.dump({
        "real_training": True,
        "model": MODEL_ID,
        "dataset_episodes": DATASET_EPISODES,
        "num_epochs": NUM_EPOCHS,
        "num_generations": NUM_GENERATIONS,
    }, f, indent=2)
print(f'Wrote training sentinel: {summary_path}')

In [ ]:
#@title 6. Evaluate trained model (LoRA adapter rollouts)

import sys, os, json
sys.path.insert(0, '/content/openenv-agent-gym')
os.chdir('/content/openenv-agent-gym')

from training.rollout_collection import collect
from training.evaluation import EvaluationSuite
from training.trajectory import TrajectoryDataset

# Hard gate: baselines must exist from cell 3
for d in ['data/baseline/random', 'data/baseline/heuristic']:
    if not os.path.exists(d) or not os.listdir(d):
        raise RuntimeError(f"Baseline missing: {d}. Re-run cell 3.")

# Hard gate: training sentinel must exist from cell 5
sentinel = 'training/grpo-unsloth-output/training_summary.json'
if not os.path.exists(sentinel):
    raise RuntimeError(f"No training sentinel at {sentinel}. Re-run cell 5 (training did not complete).")

# Hard gate: adapter must exist
adapter_dir = 'training/grpo-unsloth-output'
if not os.path.exists(os.path.join(adapter_dir, 'adapter_config.json')):
    raise RuntimeError(
        f"No adapter_config.json in {adapter_dir}. "
        "Training did not produce a LoRA adapter — check cell 5 output."
    )

# Use the same base model you trained
with open(sentinel, 'r') as f:
    sentinel_data = json.load(f)
base_model = sentinel_data.get('model')
if not base_model:
    raise RuntimeError(f"Sentinel missing model field: {sentinel}")

print('=' * 70)
print('Collecting rollouts with policy=adapter')
print(f'  base_model  : {base_model}')
print(f'  adapter_path: {adapter_dir}')
print('=' * 70)

trained_ds = collect(
    episodes=10,
    policy='adapter',
    adapter_path=adapter_dir,
    base_model=base_model,
    output_dir='data/trained',
    seed=123,
)

random_baseline = TrajectoryDataset.load_dir('data/baseline/random')
heuristic_baseline = TrajectoryDataset.load_dir('data/baseline/heuristic')
print(f'Loaded baselines: random={len(random_baseline)} heuristic={len(heuristic_baseline)}')

report = EvaluationSuite.full_report(trained_ds, reference=heuristic_baseline, label='adapter_trained')
print('\n=== Adapter-Trained Model Report ===')
print(json.dumps(report, indent=2, default=str))

In [ ]:
#@title 7. Generate monitoring plots

import sys, os
sys.path.insert(0, '/content/openenv-agent-gym')
os.chdir('/content/openenv-agent-gym')

import matplotlib  # noqa: F401
from training.monitoring import TrainingMonitor

monitor = TrainingMonitor(window=10)

# Hard gate: all three data sources must exist
required_dirs = ['data/baseline/random', 'data/baseline/heuristic', 'data/trained']
for d in required_dirs:
    if not os.path.exists(d):
        raise RuntimeError(f"Required data directory missing: {d}")
    monitor.ingest_dir(d)
    print(f'[OK] Ingested {d}')

monitor.print_summary()

os.makedirs('monitoring', exist_ok=True)
monitor.save_json('monitoring/report.json')
monitor.plot('monitoring', title='Training Progress')
print('\nMonitoring plots saved to monitoring/')

In [ ]:
#@title 8. Generate comparison plots (baseline vs trained)

import sys, os
sys.path.insert(0, '/content/openenv-agent-gym')
os.chdir('/content/openenv-agent-gym')

from training.plot_rewards import plot_compare

input_dirs = ['data/baseline/random', 'data/baseline/heuristic', 'data/trained']
labels = ['Random Baseline', 'Heuristic Baseline', 'GRPO Trained']

for d in input_dirs:
    if not os.path.exists(d):
        raise RuntimeError(f"Required directory missing: {d}. Run earlier cells first.")

os.makedirs('monitoring', exist_ok=True)
plot_compare(
    input_dirs=input_dirs,
    labels=labels,
    output_path='monitoring/full_comparison.png',
    title='Before/After: Baseline vs GRPO Trained',
)
print(f'[OK] Comparison plot generated with {len(input_dirs)} datasets -> monitoring/full_comparison.png')

In [ ]:
#@title 9. Download results

import sys, os, json, shutil
sys.path.insert(0, '/content/openenv-agent-gym')
os.chdir('/content/openenv-agent-gym')

try:
    from google.colab import files
    COLAB_AVAILABLE = True
    print('Google Colab environment detected')
except ImportError:
    COLAB_AVAILABLE = False
    print('Not in Google Colab â€” files will be listed for manual download')

os.makedirs('results', exist_ok=True)

# Package artifacts â€” must exist (earlier cells would have failed otherwise)
artifacts = [
    ('monitoring', 'results/meta-agent-gym-results'),
    ('data/trained', 'results/meta-agent-gym-trained'),
    ('training/grpo-unsloth-output', 'results/meta-agent-gym-model'),
]
created = []
for src, dest in artifacts:
    if not os.path.exists(src):
        raise RuntimeError(f"Expected artifact missing: {src}. Did earlier cells complete?")
    shutil.make_archive(dest, 'zip', src)
    zip_path = f'{dest}.zip'
    created.append(zip_path)
    print(f'[OK] {zip_path}')

# Write a summary that includes the real-training sentinel content so reviewers
# can see whether this run was real.
sentinel_path = 'training/grpo-unsloth-output/training_summary.json'
sentinel = json.load(open(sentinel_path)) if os.path.exists(sentinel_path) else {"real_training": False}

summary = {
    'notebook': 'train_colab.ipynb',
    'training_sentinel': sentinel,
    'zips': created,
}
with open('results/summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('\n' + '=' * 50)
if COLAB_AVAILABLE:
    print('Downloading to your machine...')
    for path in created + ['results/summary.json']:
        if os.path.exists(path):
            print(f'[...] {path}')
            files.download(path)
else:
    print('Files ready (manual download):')
    for path in created + ['results/summary.json']:
        if os.path.exists(path):
            print(f'  {path}')

print('\n[DONE] Notebook execution complete.')